# Consistency Trajectory Models (CTM)

[![arXiv](https://img.shields.io/badge/arXiv-2310.02279-<COLOR>.svg)](https://arxiv.org/abs/2310.02279) [![GitHub Repo stars](https://img.shields.io/github/stars/sony/ctm?style=social) ](https://github.com/sony/ctm)

## 📖 Introduction

[A Consistency Trajectory Model (CTM)](https://consistencytrajectorymodel.github.io/CTM/) is a novel generative model that generalizes existing approaches like [Consistency Models](https://github.com/openai/consistency_models) and score-based diffusion models. It trains a single neural network to learn the entire trajectory of the Probability Flow Ordinary Differential Equation (ODE) in a diffusion process. This allows the model to output both scores (gradients of log-density) and enable movement between any points along the ODE solution curve in a single forward pass. By learning the full ODE trajectory, CTM offers fast, high-quality, and flexible sampling, overcoming limitations of previous approaches. The integration of GAN loss for performance enhancement and the introduction of γ-sampling for controlling stochasticity and semantic preservation are particularly noteworthy contributions.

<img src="assets/Picture3.png" alt="CTM" width="300" height="200">

### Key Idea

_Learn a model that directly parameterizes the entire probability flow ODE trajectory of diffusion models, enabling efficient sampling at arbitrary noise levels while preserving trajectory consistency. Essentially, CTM learns to represent the entire trajectory of the diffusion process, allowing for flexible and efficient sampling and generation._

### Definition of Consistency Trajectory Models (CTM)

CTM estimates the "anytime-to-anytime jump" along the PF ODE, meaning it can predict both infinitesimally small jumps (related to the score function) and long jumps (the integral over any time horizon). This provides increased flexibility during inference.

The true solution of the PF ODE from initial time $t$ to final time $s \leq t $ is defined as:

$G(x_t, t, s) := x_t + \int_{t}^{s} \frac{x_u - E[x|x_u]}{u} \, du $

The trajectory function of models:
- teacher trajectory:= $G_{sg}(\theta) (G_{sg}(\theta)(TeacherSolver(x_t,t,u),u,s),s,0)$
- student trajectory:= $G_{sg}(\theta) (G_{sg}(\theta)((x_t,t,s),s,0))$

### Key features and Contributions

- High-quality and fast sampling (single step or with a few steps)
- Flexible Sampling Schemes (γ-sampling) - Controllable Semantic Information in Generation (inpainting)
- Clear Trade-off between Speed and Quality
- Student model beats teacher model

<img src="assets/ctm_gamma.svg" alt="γ-sampling" width=600 height=300> 
<img src="assets/Picture4.png" alt="CTM" width="700" height=600">


## Training Methodology:

![CTM Training](assets/ctm_model.svg)

CTMs (the "student model") are typically trained by distilling knowledge from a pre-trained diffusion model (the "teacher model").

- The CTM is trained to replicate the trajectory of the teacher model's ODE solution
- The training involves defining an initial state (x_t), a start time (t), and an end time (s). An intermediate time (u) is used to split the teacher's trajectory.
- Knowledge Distillation: TThe model then learns to map from x_t to x_s directly, while also ensuring consistency by comparing this direct path to a path that goes from x_t to x_u using the teacher model and then from x_u to x_s using the CTM. 
- Consistency Loss: The CTM aims to make the point reached by applying only the CTM from t to s consistent with the point reached by applying the teacher from t to u and then the CTM from u to s. "CTM tries to match these two points."
- Perceptual Alignment: To improve the perceived quality of generations, the trajectory is also extended back to time 0 (the clean image).
- GAN Loss for Quality Improvement ("Student Beats Teacher"): CTM incorporates a GAN loss by directly comparing the output of the CTM at time 0 (x_student) with the true data (x_true). This allows the CTM's performance to exceed that of the teacher model, overcoming the limitations of purely distilling the teacher's knowledge which may have inherent errors. 

### γ-sampling in CTMs
γ-sampling is a new sampling method enabled by CTM's knowledge of the entire solution trajectory. γ is a hyperparameter that allows users to adjust the stochasticity of the sampling process. 

![CTM](assets/cat.png)

- When γ=1, the sampling is fully stochastic and similar to baseline methods. 
- When γ=0, the sampling is deterministic and unique to CTM. 

By setting γ to values between 0 and 1, users can introduce a desired level of stochasticity. This flexibility allows for controlling the degree to which the generated image retains semantic information from the initial state, making it useful for tasks like inpainting where semantic preservation is important, or for generating diverse images where meaning change is desired.

### Advantages over prior Distillation models

- More efficient learning of the entire probability flow trajectory
- Better preservation of trajectory consistency
- Higher Single-Step Performance
- More flexible control over the generation process
- Enhanced distillation of diffusion model knowledge

## Cross domain application of CTM

Consistency Trajectory Models provide a powerful framework for generative modeling across various domains. While we've explored the theoretical foundations and applications of CTM in image domain, the concept generalizes beyond image domain, like audio. For example - SoundCTM.

Sound generation represents an ideal application domain for CTM's unique capabilities. The temporal nature of audio, with its complex frequency relationships and phase coherence requirements, benefits tremendously from CTM's trajectory-based approach. 

#### SoundCTM resources

- https://arxiv.org/pdf/2405.18503
- https://github.com/sony/soundctm


## CTM - Author's words:
[![Youtube](https://i.ytimg.com/vi/Bp2t8IFmDGU/hq720.jpg?sqp=-oaymwEnCNAFEJQDSFryq4qpAxkIARUAAIhCGAHYAQHiAQoIGBACGAY4AUAB&rs=AOn4CLCuGZOY3BOVZAiQAbZT6KzU2nZQ3A)](https://www.youtube.com/watch?v=Bp2t8IFmDGU&pp=ygVMQ1RNOiBBZHZhbmNlZCBTaW5nbGUtU3RlcCBESWZmdXNpb24gTW9kZWwgZm9yIEZhc3QgYW5kIEhpZ2gtUXVhbGl0eSBTYW1wbGluZw%3D%3D)


### References

[1] Yang, Y., et al. (2023). Consistency Trajectory Models: Learning Probability Flow ODE Trajectory of Diffusion. arXiv.org. https://arxiv.org/abs/2310.02279

[2] Song, Y., Dhariwal, P., Chen, M., & Sutskever, I. (2023). Consistency models. arXiv.org. https://arxiv.org/abs/2303.01469

[3] Karras, T., Aittala, M., Aila, T., & Laine, S. (2022). Elucidating the design space of diffusion-based Generative Models. arXiv.org. https://arxiv.org/abs/2206.00364

## 🛠️ Setup

### Imports

In [ ]:
import torch as th
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

# Importing all the classes for distillation training from other scripts

from utils import *
from distillation_utils import *
from model import *
import argparse
import sys  

## 🧠  Implementation

### Download and extract the dataset

In [ ]:
!gdown https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz && tar -xf cifar-10-python.tar.gz

### Unpickle and save the dataset

In [ ]:
import pickle
import os
from PIL import Image

def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

def save_cifar10_as_images(cifar_dir, output_dir):
    # Load class names
    meta = unpickle(os.path.join(cifar_dir, 'batches.meta'))
    class_names = [label.decode('utf-8') for label in meta[b'label_names']]
    
    # Create directories for each class
    for class_name in class_names:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)
    
    # Process training data
    for batch_id in range(1, 6):
        batch_file = os.path.join(cifar_dir, f'data_batch_{batch_id}')
        batch_data = unpickle(batch_file)
        images = batch_data[b'data']
        labels = batch_data[b'labels']
        
        for i, (image, label) in enumerate(zip(images, labels)):
            # Reshape and convert to RGB image
            image = image.reshape(3, 32, 32).transpose(1, 2, 0)
            img = Image.fromarray(image)
            # Save image
            img_path = os.path.join(output_dir, class_names[label], f'train_batch{batch_id}_{i}.png')
            img.save(img_path)
    
    # Process test data
    test_batch = unpickle(os.path.join(cifar_dir, 'test_batch'))
    test_images = test_batch[b'data']
    test_labels = test_batch[b'labels']
    
    for i, (image, label) in enumerate(zip(test_images, test_labels)):
        # Reshape and convert to RGB image
        image = image.reshape(3, 32, 32).transpose(1, 2, 0)
        img = Image.fromarray(image)
        # Save image
        img_path = os.path.join(output_dir, class_names[label], f'test_{i}.png')
        img.save(img_path)

# Usage
cifar_dir = 'cifar-10-batches-py'  # Path to your extracted CIFAR-10 directory
output_dir = 'cifar10_images'      # Where to save the images in folder structure
save_cifar10_as_images(cifar_dir, output_dir)

#### Initialize Arguments

In [ ]:
MODEL_FLAGS = "--batch_size=128 --microbatch=8 --gan_different_augment=True --start_ema=0.9999 --save_interval=10000 --eval_interval=1000 --eval_fid=True --eval_similarity=True --check_dm_performance=True --compute_ema_fids=True --gan_fake_inner_type=model --gan_fake_outer_type=target_model_sg --gan_training=True --g_learning_period=2 --self_learn=True --ref_path=./cifar10-32x32.npz --data_dir=cifar10_images --gpu_usage=True"
CKPT_FLAGS = "--out_dir /datasets/Harsha/ctm/GAN/uncond/GAN_bs_528_ema_0.9999_diff_aug/ --resume_checkpoint=./model095000.pt"

In [ ]:
def create_argparser():
    defaults = dict(data_name='cifar10')
    defaults.update(train_defaults(defaults['data_name']))
    defaults.update(model_and_diffusion_defaults(defaults['data_name']))
    defaults.update(cm_train_defaults(defaults['data_name']))
    defaults.update(ctm_train_defaults(defaults['data_name']))
    defaults.update(ctm_eval_defaults(defaults['data_name']))
    defaults.update(ctm_loss_defaults(defaults['data_name']))
    defaults.update(ctm_data_defaults(defaults['data_name']))
    defaults.update()
    parser = argparse.ArgumentParser()
    add_dict_to_argparser(parser, defaults)
    return parser

In [ ]:
# Prepend a dummy program name and split the flag strings:
sys.argv = ["notebook"] + MODEL_FLAGS.split() + CKPT_FLAGS.split()

# Now your existing argument parsing code should work:
args = create_argparser().parse_args()

### Modules

#### DataModule

In [ ]:
batch_size = args.batch_size
data = load_data(
    args=args,
    data_name=args.data_name,
    data_dir=args.data_dir,
    batch_size=batch_size,
    image_size=args.image_size,
    num_workers=args.num_workers,
)

In [ ]:
if th.cuda.is_available():
    dev =  th.device("cuda")
    print(dev)
else:
    dev = th.device("cpu")

In [ ]:

class KarrasDenoiser_custom(KarrasDenoiser):
    """This class consists of all the proposed concepts related to CTM
        
    """
    def __init__(
        self,
        args,
        schedule_sampler,
        diffusion_schedule_sampler,
        feature_extractor=None,
        discriminator_feature_extractor=None,
    ):
        self.args = args
        self.schedule_sampler = schedule_sampler
        self.diffusion_schedule_sampler = diffusion_schedule_sampler
        self.feature_extractor = feature_extractor
        self.discriminator_feature_extractor = discriminator_feature_extractor
        self.num_timesteps = args.start_scales
        self.dist = nn.MSELoss(reduction='none')

    def get_num_heun_step(self, start_scales=-1, num_heun_step=-1, num_heun_step_random=None, heun_step_strategy='', time_continuous=None):
        
        # Random heun steps = True, Time continuous = False, Heun step strategy = uniform
        num_heun_step = np.random.randint(1,1+self.args.num_heun_step)
        return num_heun_step

    def ctm_losses(
        self,
        step,
        model,
        x_start,
        model_kwargs=None,
        target_model=None,
        noise=None,
        discriminator=None,
        init_step=0,
        ctm=True,
        num_heun_step=-1,
        gan_num_heun_step=-1,
        diffusion_training_=False,
        gan_training_=False,
    ):
        if model_kwargs is None:
            model_kwargs = {}
        if noise is None:
            noise = th.randn_like(x_start)

        # Getting the timesteps for s and t
        dims = x_start.ndim
        s = None
        terms = {}

        # Get the number of heun step: 
        # This step selects random heun step OR timestep t
        num_heun_step = self.get_num_heun_step(num_heun_step=self.args.num_heun_step)
        
        # Get the indexes for timestep
        indices, _ = self.schedule_sampler.sample_t(x_start.shape[0], x_start.device, num_heun_step,
                                                    self.args.time_continuous)
        # Get the EDM mapped t timestep 
        t = self.get_t(indices)

        # Get the EDM mapped u timestep --> only used with teacher
        t_dt = self.get_t(indices + num_heun_step)
        if ctm:
            # Get the index for s timestep
            new_indices = self.schedule_sampler.sample_s(self.args, x_start.shape[0], x_start.device, indices,
                                                         num_heun_step, self.args.time_continuous,
                                                         N=self.args.start_scales)
            # Get the EDM mapped s timestep 
            s = self.get_t(new_indices)
        
        # Add noise to image
        x_t = x_start + noise * append_dims(t, dims)
        
        dropout_state = th.get_rng_state()
        th.set_rng_state(dropout_state)

        # Get the output/estimate of student model
        if self.args.ctm_training:
            # The student model tries to predict denoisified version of x_t at timestep s given timestep t
            ctm_estimate = self.get_ctm_estimate(x_t, t, s, model, target_model, ctm=ctm,
                                                 outer_type=self.args.ctm_estimate_outer_type,
                                                 **model_kwargs)
        
        # If Adversarial training is enabled (gan_training = True)
        # we update Generator only at certain frequencies decided by argument g_learning_period
        # Generator update stage
        #   If Gan training is enabled
        #       loss = CTM loss + DSM loss + Generator loss(If gan training is enabled)
        #   else
        #       loss = CTM loss + DSM loss
        if step % self.args.g_learning_period == 0 or not self.args.gan_training:
            x_t_dt = self.heun_solver(target_model, x_t, indices, dims, t, t_dt, ctm=ctm, num_step=num_heun_step,
                                        **model_kwargs).detach()
            ctm_target = self.get_ctm_target(x_t_dt, t_dt, s, model, target_model, ctm=ctm,
                                                inner_type=self.args.ctm_target_inner_type, **model_kwargs)

            snrs = self.get_snr(t)
            weights = get_weightings(self.args.weight_schedule, snrs, self.args.sigma_data, t, s, self.args.weight_schedule_multiplier)

            terms["consistency_loss"] = self.get_CTM_loss(ctm_estimate, ctm_target, weights, step - init_step,)

            terms['denoising_loss'] = self.get_DSM_loss(model, x_start, model_kwargs,
                                                                terms["consistency_loss"] if self.args.ctm_training else None,
                                                                step, init_step)

            if self.args.gan_training and step - init_step >= self.args.discriminator_start_itr:
                if gan_training_:
                    gan_x_t, gan_t, gan_t_dt, gan_s, _, _ = self.get_gan_time(x_start, noise, x_t, t, t_dt, s, indices,
                                                                              num_heun_step, gan_num_heun_step)
                    gan_fake = self.get_gan_fake(ctm_estimate, gan_x_t, gan_t, gan_t_dt, gan_s, model, target_model, ctm,
                                                 step - init_step, **model_kwargs)
                    terms['d_loss'] = self.get_GAN_loss(model, fake=gan_fake,
                                                                  consistency_loss=terms["consistency_loss"],
                                                                  discriminator=discriminator,
                                                                  step=step, init_step=init_step)
        # Discriminator update stage
        # loss = Adversarial GAN loss
        else:
            gan_x_t, gan_t, gan_t_dt, gan_s, gan_indices, gan_num_heun_step = \
                self.get_gan_time(x_start, noise, x_t, t, t_dt, s, indices, num_heun_step, gan_num_heun_step)
            gan_real = self.get_gan_real(x_start, gan_x_t, gan_t, gan_t_dt, gan_s, gan_indices, dims, gan_num_heun_step,
                                         model, target_model, ctm, step - init_step, **model_kwargs)
            gan_fake = self.get_gan_fake(ctm_estimate, gan_x_t, gan_t, gan_t_dt, gan_s, model, target_model, ctm,
                                         step - init_step, **model_kwargs)
            terms['d_loss'] = self.get_GAN_loss(model, fake=gan_fake, real=gan_real,
                                                learn_generator=False, discriminator=discriminator,
                                                step=step, init_step=init_step, **model_kwargs)
        return terms



In [ ]:
def create_model_and_diffusion(args, feature_extractor=None, discriminator_feature_extractor=None, teacher=False):
    """Creates Model architecture skeleton and distillation class(Karras denoiser)"""
    schedule_sampler = create_named_schedule_sampler(args, args.schedule_sampler, args.start_scales)
    diffusion_schedule_sampler = create_named_schedule_sampler(args, args.diffusion_schedule_sampler, args.start_scales)
    model = EDMPrecond_CTM(img_resolution=args.image_size, img_channels=3,
                            label_dim=1000 if args.data_name.lower() == 'imagenet64' else 10 if args.class_cond else 0, use_fp16=args.use_fp16,
                            sigma_min=args.sigma_min, sigma_max=args.sigma_max,
                            sigma_data=args.sigma_data, model_type='SongUNet' if args.data_name.lower() == 'cifar10' else 'DhariwalUNet',
                            teacher=teacher, teacher_model_path=args.teacher_model_path or args.model_path,
                            training_mode=args.training_mode, arch='ddpmpp' if args.data_name.lower() == 'cifar10' else 'adm',
                            linear_probing=args.linear_probing)

    diffusion = KarrasDenoiser(
        args=args, schedule_sampler=schedule_sampler,
        diffusion_schedule_sampler=diffusion_schedule_sampler,
        feature_extractor=feature_extractor,
        discriminator_feature_extractor=discriminator_feature_extractor,
    )
    return model, diffusion

In [ ]:
configure(args, dir=args.out_dir)

In [ ]:
ema_scale_fn = create_ema_and_scales_fn( 
        target_ema_mode=args.target_ema_mode,
        start_ema=args.start_ema,
        scale_mode=args.scale_mode,
        start_scales=args.start_scales,
        end_scales=args.end_scales,
        total_steps=args.total_training_steps,
        distill_steps_per_iter=args.distill_steps_per_iter,
    )

# Load Feature Extractor
feature_extractor = load_feature_extractor(args, eval=True)

# Load Discriminator
discriminator, discriminator_feature_extractor = load_discriminator_and_d_feature_extractor(args)


# Load Model
model, diffusion = create_model_and_diffusion(args, feature_extractor, discriminator_feature_extractor)
model.to(dev)

## 🚀 Training

### Run Training

In [ ]:
model.train()
teacher_model = None # stand-alone setup training

In [ ]:
target_model, _ = create_model_and_diffusion(args)

target_model.to(dev)
target_model.train()
for dst, src in zip(target_model.parameters(), model.parameters()):
    dst.data.copy_(src.data)

if args.use_fp16:
    target_model.convert_to_fp16()
if args.edm_nn_ncsn:
    target_model.model.map_noise.freqs = teacher_model.model.model.map_noise.freqs
print(target_model)

In [ ]:
CTMTrainLoop(
    model=model,
    target_model=target_model,
    teacher_model=teacher_model,
    discriminator=discriminator,
    ema_scale_fn=ema_scale_fn,
    diffusion=diffusion,
    data=data,
    batch_size=batch_size,
    args=args,
).run_loop()

## Sampling

In [ ]:
def create_argparser():

    defaults = dict(
        generator="determ",
        eval_batch=16,
        sampler="heun",
        s_churn=0.0,
        s_tmin=0.0,
        s_tmax=float("inf"),
        s_noise=1.0,
        sampling_steps=40,
        model_path="",
        eval_seed=42,
        save_format='png',
        stochastic_seed=False,
        data_name='cifar10',
        ind_1=0,
        ind_2=0,
        gamma=0.5,
    )
    defaults.update(train_defaults(defaults['data_name']))
    defaults.update(model_and_diffusion_defaults(defaults['data_name']))
    defaults.update(cm_train_defaults(defaults['data_name']))
    defaults.update(ctm_train_defaults(defaults['data_name']))
    defaults.update(ctm_eval_defaults(defaults['data_name']))
    defaults.update(ctm_loss_defaults(defaults['data_name']))
    defaults.update(ctm_data_defaults(defaults['data_name']))
    parser = argparse.ArgumentParser()
    add_dict_to_argparser(parser, defaults)
    return parser

In [ ]:
MODEL_FLAGS="--start_ema=0.999 --save_check_period=1000 --eval_interval=5000 --eval_fid=True --eval_similarity=False --check_dm_performance=False --compute_ema_fids=True --gan_fake_inner_type=model --gan_fake_outer_type=target_model_sg --gan_training=True --g_learning_period=2 "
CKPT_FLAGS = "--out_dir=./ --model_path=/workspace/ctm/ctm-cifar10/GAN/uncond/GAN_bs_528_ema_0.9999_diff_aug/target_model095090.pt --eval_num_samples=8 --batch_size=8  --ind_1=1 --ind_2=0 --class_cond=False  --sampler=exact --sampling_steps=100 --device_id=1"

In [ ]:
# Prepend a dummy program name and split the flag strings:
sys.argv = ["notebook"] + MODEL_FLAGS.split() + CKPT_FLAGS.split()

# Now your existing argument parsing code should work:
args = create_argparser().parse_args()
configure(args, dir=args.out_dir)

In [ ]:
model.load_state_dict(
    load_state_dict(args.model_path, map_location=dev)
)

model.to(dev)

model.convert_to_fp16()
model.eval()

In [ ]:
# using stochastic seed
args.eval_seed = np.random.randint(1000000)
generator = get_generator(args.generator, args.eval_num_samples, args.eval_seed)

step = args.model_path.split('.')[-2][-6:]

In [ ]:
ts = []
# sampler used is exact
out_dir = os.path.join(args.out_dir, f'{args.training_mode}_{args.sampler}_sampler_{args.sampling_steps}_steps_{step}_itrs_model_ema_{"".join([str(i) for i in ts])}')

In [ ]:
import matplotlib.pyplot as plt

os.makedirs(out_dir, exist_ok=True)
itr = 0
eval_num_samples = 0
while itr * args.batch_size < args.eval_num_samples:
    x_T = generator.randn(
        *(args.batch_size, args.in_channels, args.image_size, args.image_size),
        device=dev) * args.sigma_max
    print("x_T: ", x_T[0][0][0][0])
    current = time.time()
    model_kwargs = {}

    with th.no_grad():
        x = karras_sample(
            diffusion=diffusion,
            model=model,
            shape=(args.batch_size, args.in_channels, args.image_size, args.image_size),
            steps=args.sampling_steps,
            model_kwargs=model_kwargs,
            device=dev,
            clip_denoised=False if args.data_name in ['church'] else True if args.training_mode=='edm' else args.clip_denoised,
            sampler=args.sampler,
            sigma_min=args.sigma_min,
            sigma_max=args.sigma_max,
            s_churn=args.s_churn,
            s_tmin=args.s_tmin,
            s_tmax=args.s_tmax,
            s_noise=args.s_noise,
            generator=None,
            ts=ts,
            teacher = True if args.training_mode == 'edm' else False,
            clip_output=args.clip_output,
            ctm=True if args.training_mode.lower() == 'ctm' else False,
            x_T=x_T if args.stochastic_seed == False else None,
            ind_1=args.ind_1,
            ind_2=args.ind_2,
            gamma=args.gamma,
        )

    sample = ((x + 1) * 127.5).clamp(0, 255).to(th.uint8)
    sample = sample.permute(0, 2, 3, 1)
    sample = sample.contiguous()

    sample = sample.cpu().detach()
    print(f"{(itr-1) * args.batch_size} sampling complete...")
    r = np.random.randint(1000000)

    # save format is png
    print("x range: ", x.min(), x.max())
    print(out_dir)
    nrow = int(np.sqrt(sample.shape[0]))
    image_grid = make_grid((x + 1.) / 2., nrow, padding=2)
    
    # Convert the image grid to a format suitable for imshow
    np_image = image_grid.permute(1, 2, 0).cpu().numpy()  # CxHxW -> HxWxC
    
    plt.figure(figsize=(4, 4))
    plt.axis('off')
    plt.imshow(np_image)
    plt.show()

    eval_num_samples += sample.shape[0]
    print(f"sample {eval_num_samples} time {time.time() - current} sec")
    itr += 1

print("sampling complete")

## Acknowledgment

This notebook on the Consistency Trajectory Model (CTM) is inspired by the work on the [Consistency Model (CM)](https://colab.research.google.com/github/Kinyugo/consistency_models/blob/main/notebooks/consistency_models_training_example.ipynb). 